# 0. Environment

In [4]:
import os
import pandas as pd
import csv
import json
from pathlib import Path
from collections import defaultdict
import re
import unicodedata

import time

In [ ]:
os.chdir("C:\\Users\\yatsyk\\Documents\\NextCloud_Synchro\\passim_BoH\\")

# 1. Get familiar with file structure

In [ ]:
# get the keys
with open("deep-jsonl\\parts_0006.jsonl", "r", encoding="utf-8") as f:
    line = f.readline()
    obj = json.loads(line)
    print(obj.keys())


In [ ]:
# get biblissima ids from & file
with open("deep-jsonl\\parts_0006.jsonl", "r", encoding="utf-8") as f:
    line = f.readline()
    obj = json.loads(line)
    print(obj.get("biblissima_id"))

# 2. Extract 196 mss

In [ ]:
jsonl_dir = Path(r"C:\datasets\deep-jsonl")
#csv_path = Path(r"04.CoMMa\02.metadata\livres-d'heures-comma-horae.csv")
csv_path = Path(r"04.CoMMa\02.metadata\196_arkindex_Comma_URL_filled.csv")

In [9]:
df = pd.read_csv(csv_path, sep=",", encoding="utf-8")
df

,Manuscript H-ID,bbma_data_entity,ID Arkindex,URL CoMMA,Structural annotation,partially annotated
0,1356,https://data.biblissima.fr/entity/Q241713,6150a30a-bff0-4672-96c6-9c19896cdc5e,https://comma.inria.fr/doc/https%253A%252F%252...,False,False
1,1429,https://data.biblissima.fr/entity/Q235097,77fcbfe1-faf8-4316-8f32-0155ed3a60c5,https://comma.inria.fr/doc/https%253A%252F%252...,False,True
2,1430,https://data.biblissima.fr/entity/Q235098,034c35ae-17e3-4e51-9821-8cf1662b1253,https://comma.inria.fr/doc/https%253A%252F%252...,False,True
3,1642,https://data.biblissima.fr/entity/Q235484,73e26638-6275-41b9-9502-39e27f01d93f,https://comma.inria.fr/doc/https%253A%252F%252...,False,False
4,1829,https://data.biblissima.fr/entity/Q203702,b3b9759b-56e1-460d-b766-8cee25d6aaa5,https://comma.inria.fr/doc/https%253A%252F%252...,False,True
...,...,...,...,...,...,...
191,3118,https://data.biblissima.fr/entity/Q209026,5fe2eace-2177-4fa0-8c84-2b94b3d88c00,https://comma.inria.fr/doc/https%253A%252F%252...,False,True
192,3119,https://data.biblissima.fr/entity/Q209044,a9a01b20-b6cc-43e1-b3cd-4b57bb3fc270,https://comma.inria.fr/doc/https%253A%252F%252...,False,True
193,3355,https://data.biblissima.fr/entity/Q209755,c0b8fea9-df2b-4518-a55e-5337fa54ce13,https://comma.inria.fr/doc/https%253A%252F%252...,False,True
194,4496,https://data.biblissima.fr/entity/Q211784,bb70bf8b-f365-47a2-9795-37d7c788518c,https://comma.inria.fr/doc/https%253A%252F%252...,False,True


In [10]:
output_dir = Path(r"04.CoMMa\00.data\00.jsonls")
output_dir.mkdir(parents=True, exist_ok=True)

# read the biblissima ids from csv
df = pd.read_csv(csv_path, sep=",", encoding="utf-8")

bbma_ids = set(
    df["bbma_data_entity"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"✅ bbma_data_entity: {len(bbma_ids)} entries")


def extract_short_id(full_id: str) -> str:
    """Extract short id from biblissima_id URL."""
    return full_id.rstrip("/").split("/")[-1]


# existing output files
existing_files = {p.stem for p in output_dir.glob("*.jsonl")}
print(f"Already existing files: {len(existing_files)}")

# ids for which output files already exist
ids_to_skip = {
    full_id for full_id in bbma_ids
    if extract_short_id(full_id) in existing_files
}

# ids that still need to be searched and written
ids_to_find = bbma_ids - ids_to_skip

print(f"IDs skipped because file already exists: {len(ids_to_skip)}")
print(f"IDs to search: {len(ids_to_find)}")


# cache open files
open_files = {}

found_ids = set()
created_ids = set()


def get_file_handle(full_id: str):
    short_id = extract_short_id(full_id)
    file_path = output_dir / f"{short_id}.jsonl"

    if short_id not in open_files:
        print(f"Creating new file: {file_path.name}")
        open_files[short_id] = open(file_path, "w", encoding="utf-8")
        created_ids.add(full_id)

    return open_files[short_id]


# go through jsonl files and extract matching objects
jsonl_files = list(jsonl_dir.rglob("*.jsonl"))

processed = 0
candidate_lines = 0
matched = 0

start = time.time()

for file in jsonl_files:
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            processed += 1

            # fast pre-filter: skip JSON parsing if none of the target ids
            
            id_pattern = re.compile(
                "|".join(re.escape(bid) for bid in ids_to_find)
            )

            if not id_pattern.search(line):
                continue

            candidate_lines += 1

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            biblissima_id = obj.get("biblissima_id")
            if not biblissima_id:
                continue

            biblissima_id = str(biblissima_id).strip()

            if biblissima_id in ids_to_find:
                matched += 1
                found_ids.add(biblissima_id)

                out_f = get_file_handle(biblissima_id)
                out_f.write(json.dumps(obj, ensure_ascii=False) + "\n")

            if processed % 100000 == 0:
                elapsed = time.time() - start
                print(
                    print(f"{processed:,} lines processed in {elapsed/60:.1f} min"),
                    f"{candidate_lines} candidate lines | "
                    f"{matched} matches | "
                    f"{elapsed:.1f}s"
                )


# close all open files
for f in open_files.values():
    f.close()

missing_ids = ids_to_find - found_ids

print("\nDone!")
print(f"IDs requested: {len(bbma_ids)}")
print(f"IDs skipped because file already existed: {len(ids_to_skip)}")
print(f"IDs searched: {len(ids_to_find)}")
print(f"Processed lines: {processed}")
print(f"Candidate lines parsed as JSON: {candidate_lines}")
print(f"Matched rows: {matched}")
print(f"Unique matched IDs: {len(found_ids)}")
print(f"Created files: {len(created_ids)}")
print(f"Missing IDs among searched IDs: {len(missing_ids)}")

✅ bbma_data_entity: 196 entries
Already existing files: 163
IDs skipped because file already exists: 163
IDs to search: 33
Creating new file: Q68520.jsonl
Creating new file: Q58724.jsonl
Creating new file: Q261036.jsonl
Creating new file: Q46285.jsonl
Creating new file: Q109589.jsonl
Creating new file: Q109836.jsonl
Creating new file: Q70010.jsonl
Creating new file: Q58827.jsonl
Creating new file: Q67077.jsonl
Creating new file: Q241742.jsonl
Creating new file: Q66038.jsonl
Creating new file: Q241751.jsonl
Creating new file: Q118363.jsonl
Creating new file: Q48872.jsonl
Creating new file: Q241713.jsonl
Creating new file: Q262274.jsonl
Creating new file: Q67249.jsonl
Creating new file: Q106616.jsonl
Creating new file: Q58836.jsonl
Creating new file: Q69704.jsonl
Creating new file: Q209028.jsonl
Creating new file: Q47874.jsonl
Creating new file: Q47860.jsonl
Creating new file: Q242395.jsonl
Creating new file: Q58747.jsonl
Creating new file: Q109835.jsonl
Creating new file: Q60184.jsonl
C

In [ ]:
all_ids_in_json = set()

for file in jsonl_files:
    with open(file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i > 10000:
                break
            obj = json.loads(line)
            bid = obj.get("biblissima_id")
            if bid:
                all_ids_in_json.add(bid)

print("Sample JSON IDs:", list(all_ids_in_json)[:10])
print("Intersection:", len(all_ids_in_json & ids_to_find))

# 3. Extract transcriptions from jsonl

In [12]:
input_dir = Path(r"04.CoMMa\00.data\00.jsonls")
output_dir = Path(r"04.CoMMa\00.data\01.csvs")
output_dir.mkdir(exist_ok=True)

folio_pattern = re.compile(r'(\d+[rv])(?=\.|$)')

## 3.1. Helper functions

### 3.1.1. Extract folio

In [13]:
def extract_folio(page_id: str) -> str | None:
    if not page_id:
        return None
    match = folio_pattern.search(str(page_id))
    return match.group(1) if match else None

### 3.1.2. Extract text from DefaultLines for each page

In [14]:
def extract_page_rows_from_record(record: dict) -> list[dict]:
    rows = []

    files = record.get("files", [])
    if not isinstance(files, list):
        return rows

    for page in files:
        page_id = page.get("alto", "")
        folio = extract_folio(page_id)

        text_parts = []
        zones = page.get("zones", [])

        if isinstance(zones, list):
            for zone in zones:
                if zone.get("type") != "MainZone":
                    continue

                lines = zone.get("lines", [])
                if not isinstance(lines, list):
                    continue

                for line in lines:
                    if line.get("type") == "DefaultLine":
                        content = line.get("content", "")
                        if content is not None:
                            content = str(content).strip()
                            content = unicodedata.normalize("NFC", content)
                            if content:
                                text_parts.append(content)

        rows.append({
            "page_id": page_id,
            "folio": folio,
            "text": " ".join(text_parts)
        })

    return rows

### 3.1.3. Transform jsonls to csvs

In [15]:
def process_jsonl_file(jsonl_path: Path, output_dir: Path) -> None:
    all_rows = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"JSON error in {jsonl_path.name}, line {line_num}: {e}")
                continue

            page_rows = extract_page_rows_from_record(record)
            all_rows.extend(page_rows)

    df = pd.DataFrame(all_rows, columns=["page_id", "folio", "text"])

    out_path = output_dir / f"{jsonl_path.stem}.csv"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"✅ Saved: {out_path} ({len(df)} pages)")

## 3.2. Implementation

In [16]:
jsonl_files = list(input_dir.rglob("*.jsonl"))
print(f"Found jsonl files: {len(jsonl_files)}")

for jsonl_file in jsonl_files:
    process_jsonl_file(jsonl_file, output_dir)

print("Done.")

Found jsonl files: 196
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106117.csv (153 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106514.csv (349 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106532.csv (224 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106533.csv (361 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106537.csv (240 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106540.csv (199 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106541.csv (257 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106543.csv (301 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106545.csv (545 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106616.csv (285 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q106852.csv (544 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q107627.csv (278 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q107628.csv (565 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q107656.csv (564 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q107662.csv (577 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q109291.csv (683 pages)
✅ Saved: 04.CoMMa\00.data\01.csvs\Q109292.csv (16

## 3.3. Check whether the encoding is ok

In [17]:
df = pd.read_csv("04.CoMMa\\00.data\\01.csvs\\Q106532.csv", encoding="utf-8-sig")
print(df["text"].iloc[25])

mor meus. Dremus. Gr̃o Omine ihesuxpriste fiii dei uiui pone passionem crucem et mortem tuã inter iudiciũ tuũ ⁊ animã meã nunt et in hora ex tus mei et largiri digneris bi uis mĩam et gr̃am defunitiore quiem et uemam ecclesie sancte tur patem et tontordrametno bis miseris pttorib mĩam et letitram sempiternam Qui unus et regnas deus per oĩa secula setulorumAmen Dñe exaudi Et clamor Benedica nius domino Deog̾. Eus maduitorium me um intende Domine ad adiu


## 3.4. Add page_ordering

In [18]:
csv_dir = Path("04.CoMMa/00.data/01.csvs")

for csv_path in csv_dir.glob("*.csv"):
    df = pd.read_csv(csv_path)

    df["page_ordering"] = range(1, len(df) + 1)

    df.to_csv(csv_path, index=False)